In [ ]:
!git clone https://github.com/keshav22bansal/BAKSA_IITK.git

Cloning into 'BAKSA_IITK'...
remote: Enumerating objects: 540, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 540 (delta 4), reused 8 (delta 4), pack-reused 531 (from 1)
Receiving objects: 100% (540/540), 132.04 MiB | 16.32 MiB/s, done.
Resolving deltas: 100% (271/271), done.


In [ ]:
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from collections import Counter

train_df = pd.read_csv('BAKSA_IITK/data/hinglish/train.txt', sep='\t')
test_df = pd.read_csv('BAKSA_IITK/data/hinglish/test.txt', sep='\t')
print(train_df.shape, test_df.shape)

(14000, 3) (3131, 3)


In [ ]:
train_df['text_length'] = train_df['text'].apply(lambda x: len(str(x).split()))
train_df = train_df[train_df['text_length'] > 0].reset_index(drop=True)

def clean_text(text):
    text = str(text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

train_df = train_df[train_df['clean_text'].str.len() > 0].reset_index(drop=True)
test_df = test_df[test_df['clean_text'].str.len() > 0].reset_index(drop=True)
print(train_df.shape, test_df.shape)

(13984, 5) (3130, 4)


In [ ]:
train_texts_set = set(train_df['clean_text'])
test_df_clean = test_df[~test_df['clean_text'].isin(train_texts_set)].reset_index(drop=True)
print("Clean test size:", len(test_df_clean))

Clean test size: 821


In [ ]:
!pip install sentencepiece
import sentencepiece as spm

with open('sp_train_input.txt', 'w', encoding='utf-8') as f:
    for text in train_df['clean_text']:
        f.write(text + '\n')

spm.SentencePieceTrainer.train(
    input='sp_train_input.txt',
    model_prefix='hinglish_sp',
    vocab_size=8000,
    model_type='bpe',
    character_coverage=1.0,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load('hinglish_sp.model')
vocab_size = sp.get_piece_size()
print(vocab_size)

8000


In [ ]:
MAX_LEN = 32
PAD_ID = 0

def encode_and_pad(text, sp=sp, max_len=MAX_LEN, pad_id=PAD_ID):
    ids = sp.encode(text, out_type=int)
    ids = ids[:max_len]
    ids = ids + [pad_id] * (max_len - len(ids))
    return ids

train_df['Encoded'] = train_df['clean_text'].apply(encode_and_pad)
test_df['Encoded'] = test_df['clean_text'].apply(encode_and_pad)
test_df_clean['Encoded'] = test_df_clean['clean_text'].apply(encode_and_pad)
print(len(train_df['Encoded'].iloc[0]))

32


In [ ]:
!wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.vec.gz
!gunzip cc.hi.300.vec.gz

In [ ]:
embedding_dim = 300
piece_to_id = {sp.id_to_piece(i): i for i in range(vocab_size)}

np.random.seed(42)
embedding_matrix = np.random.normal(scale=0.1, size=(vocab_size, embedding_dim))

matched_ids = set()
found = 0
with open('cc.hi.300.vec', encoding='utf-8') as f:
    next(f)
    for line in f:
        parts = line.rstrip().split(' ')
        word = parts[0]
        if word in piece_to_id:
            idx = piece_to_id[word]
            vector = np.array(parts[1:], dtype=np.float32)
            embedding_matrix[idx] = vector
            matched_ids.add(idx)
            found += 1

unmatched_stripped_to_id = {}
for piece, idx in piece_to_id.items():
    if idx not in matched_ids:
        stripped = piece.replace('▁', '')
        unmatched_stripped_to_id[stripped] = idx

with open('cc.hi.300.vec', encoding='utf-8') as f:
    next(f)
    for line in f:
        parts = line.rstrip().split(' ')
        word = parts[0]
        if word in unmatched_stripped_to_id:
            idx = unmatched_stripped_to_id[word]
            if idx not in matched_ids:
                vector = np.array(parts[1:], dtype=np.float32)
                embedding_matrix[idx] = vector
                matched_ids.add(idx)
                found += 1

print(f"Total found: {found} / {vocab_size} ({found/vocab_size*100:.1f}%)")

Total found: 6946 / 8000 (86.8%)


In [ ]:
class HinglishDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


class HinglishCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, num_classes=3,
                 num_filters=170, filter_sizes=[2,3, 4, 5], dropout=0.6, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.permute(0, 2, 1)
        pooled_results = []
        for conv in self.convs:
            conv_out = conv(embedded)
            activated = F.relu(conv_out)
            pooled = F.max_pool1d(activated, kernel_size=activated.shape[2])
            pooled = pooled.squeeze(2)
            pooled_results.append(pooled)
        concatenated = torch.cat(pooled_results, dim=1)
        dropped = self.dropout(concatenated)
        logits = self.fc(dropped)
        return logits

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df['label']
)

train_seq = torch.tensor(train_split_df['Encoded'].tolist(), dtype=torch.long)
train_lab = torch.tensor(train_split_df['label'].tolist(), dtype=torch.long)
val_seq = torch.tensor(val_split_df['Encoded'].tolist(), dtype=torch.long)
val_lab = torch.tensor(val_split_df['label'].tolist(), dtype=torch.long)
test_seq_clean = torch.tensor(test_df_clean['Encoded'].tolist(), dtype=torch.long)
test_lab_clean = torch.tensor(test_df_clean['label'].tolist(), dtype=torch.long)

train_loader = DataLoader(HinglishDataset(train_seq, train_lab), batch_size=64, shuffle=True)
val_loader = DataLoader(HinglishDataset(val_seq, val_lab), batch_size=64, shuffle=False)
test_loader_clean = DataLoader(HinglishDataset(test_seq_clean, test_lab_clean), batch_size=64, shuffle=False)

class_counts = train_split_df['label'].value_counts().sort_index().values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * len(class_counts)).to(device)

cuda


In [ ]:
model = HinglishCNN(vocab_size=vocab_size, embedding_dim=300, embedding_matrix=embedding_matrix).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(weight=class_weights)

epochs= 30
PATIENCE = 3
best_f1 = 0
epochs_no_improve = 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_sequences, batch_labels in train_loader:
        batch_sequences, batch_labels = batch_sequences.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        outputs = model(batch_sequences)
        loss_val = criterion(outputs, batch_labels)
        loss_val.backward()
        optimizer.step()
        total_loss += loss_val.item()
    avg_train_loss = total_loss / len(train_loader)



In [ ]:
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch_sequences, batch_labels in val_loader:
            batch_sequences, batch_labels = batch_sequences.to(device), batch_labels.to(device)
            outputs = model(batch_sequences)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(batch_labels.cpu().numpy())
    val_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}. Best val_f1: {best_f1:.4f}")


model.load_state_dict(best_model_state)

Epoch 30: train_loss=0.0330, val_f1=0.5222


<All keys matched successfully>

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch_sequences, batch_labels in test_loader_clean:
        batch_sequences, batch_labels = batch_sequences.to(device), batch_labels.to(device)
        outputs = model(batch_sequences)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.cpu().numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Test Accuracy: {test_acc:.4f}, Test Macro F1: {test_f1:.4f}")
print(classification_report(all_labels, all_preds, target_names=['negative','neutral','positive']))

Test Accuracy: 0.6565, Test Macro F1: 0.6559
              precision    recall  f1-score   support

    negative       0.65      0.63      0.64       226
     neutral       0.66      0.63      0.65       326
    positive       0.66      0.71      0.68       269

    accuracy                           0.66       821
   macro avg       0.66      0.66      0.66       821
weighted avg       0.66      0.66      0.66       821



In [ ]:
errors = []
for i in range(len(all_preds)):
    if all_preds[i] != all_labels[i]:
        errors.append({
            'text': test_df_clean['clean_text'].iloc[i],
            'true': all_labels[i],
            'pred': all_preds[i]
        })

print(f"Total errors: {len(errors)} / {len(all_preds)}")

label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
for e in errors[:15]:
    print(f"TRUE={label_names[e['true']]} PRED={label_names[e['pred']]}: {e['text']}")
    print()

Total errors: 282 / 821
TRUE=neutral PRED=positive: वह वह Atif ज़ंजीर सलामत है झंकार सलामत है ! थैंक यू & for your

TRUE=neutral PRED=negative: संसद की garima corruption की वजह से ज्यादा ख़राब हो रही है जनाब ..

TRUE=neutral PRED=negative: तो वह दूसरी एव्म कहा से आई ?? उसका तोह ाँस Do 19 lack एवं खा गयी ???

TRUE=neutral PRED=positive: ा पांचोली & veteran ... good joke .. with barely 2 emotions he is being branded as actor ... लोल

TRUE=positive PRED=neutral: त्यागी हैप्पी बर्थडे अदिति त्यागी भगवन से प्राथना करता हु की आपका खुदका चैनल खुल जाये और पूरा दिन a

TRUE=neutral PRED=positive: I'm scared of ! पूजा सो Happy to work with you on

TRUE=positive PRED=neutral: ऐषा ावरी चाण्डा hun its TAD07 शहकर मर गुड है

TRUE=neutral PRED=negative: बीएस इस बाबा बरखा बंगाली ड्राइवर वाले you are talking bout . बहुत पहुँचे हुए बाबा हैं .. सॉरी the .. Ministe

TRUE=positive PRED=negative: Pakistani cricket team is the most entertaining team in the world . Indian politics के बाद सबसे ज्यादा कंटेंट यही मि


In [ ]:
import os
os.makedirs('export', exist_ok=True)

# 1. Save model weights
torch.save(model.state_dict(), 'export/hinglish_cnn.pt')

# 2. Save the SentencePiece tokenizer
!cp hinglish_sp.model export/hinglish_sp.model

# 3. Save model config (so we know what architecture to rebuild when loading)
import json
config = {
    'vocab_size': vocab_size,
    'embedding_dim': 300,
    'num_classes': 3,
    'num_filters': 100,
    'filter_sizes': [3, 4, 5],
    'dropout': 0.5,
    'max_len': MAX_LEN,
    'pad_id': PAD_ID,
    'label_map': {0: 'negative', 1: 'neutral', 2: 'positive'}
}
with open('export/config.json', 'w') as f:
    json.dump(config, f, indent=2)

!ls -lh export/

total 13M
-rw-r--r-- 1 root root  268 Jul 12 11:55 config.json
-rw-r--r-- 1 root root  12M Jul 12 11:55 hinglish_cnn.pt
-rw-r--r-- 1 root root 376K Jul 12 11:55 hinglish_sp.model
